# Lesson 6. Many qubits and superposition

**Goal:** understand which space the state of many qubits lives in, where the exponential dimension comes from, and what exactly a superposition is.

## 1. The space: a tensor product

The state of $n$ qubits is a unit vector in

$$\underbrace{\mathbb{C}^2 \otimes \cdots \otimes \mathbb{C}^2}_{n} \;\cong\; \mathbb{C}^{2^n}.$$

> **This was tricky:** the dimension is $2^n$ (two to the power $n$), not $2n$. For 3 qubits it is $\mathbb{C}^8$, not $\mathbb{R}^8$ (amplitudes are complex) and not $\mathbb{C}^4$ (the basis has 8 vectors, not 4).

In a tensor product the dimensions **multiply** ($2 \cdot 2 \cdot 2 = 8$); in a direct sum they **add** ($2 + 2 + 2 = 6$). Physics uses the tensor product because measuring 3 qubits has 8 possible outcomes, and the Born rule needs a separate amplitude for each. Even a classical joint distribution of three bits needs 8 numbers, because the bits can be correlated.

| Qubits | $2n$ | $2^n$ |
|---|---|---|
| 1 | 2 | 2 |
| 2 | 4 | 4 |
| 3 | 6 | **8** |
| 10 | 20 | **1024** |
| 30 | 60 | **1 073 741 824** |

**The intuition "$n$ independent vectors in $\mathbb{C}^2$" is not wrong, just incomplete.** It describes one special case: product states $a \otimes b \otimes c$, where each qubit has its own state. Most vectors in $\mathbb{C}^{2^n}$ do not look like that (lesson 7).

## 2. The basis and the Kronecker product

The basis of $\mathbb{C}^{2^n}$ consists of products of basis vectors: $e_a \otimes e_b \otimes e_c$. The index is the number written in bits:

$$e_a \otimes e_b = e_{2a+b}, \qquad e_a \otimes e_b \otimes e_c = e_{4a+2b+c}.$$

So the label $|101\rangle$ is $e_5 \in \mathbb{C}^8$: a one at position $101_2 = 5$.

A gate acting on one qubit, with "do nothing" on the others, is a Kronecker product with identities, e.g. X on the first of three qubits:

$$X \otimes I \otimes I = \begin{pmatrix}0 & I_4\\ I_4 & 0\end{pmatrix}.$$

It swaps the top half of the vector (states with first bit 0) with the bottom half (first bit 1).

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
GREEN, DARK, LIGHT, GREY, ORANGE = "#44B749", "#1A8A1F", "#96E199", "#8A8A8A", "#E08A00"

e0 = np.array([1, 0], dtype=complex)   # label |0⟩
e1 = np.array([0, 1], dtype=complex)   # label |1⟩

def norm(v):
    return np.sqrt(np.sum(np.abs(v) ** 2))

def show(v, n=None):
    """Print a state vector with bit labels (label = position written in binary)."""
    v = np.asarray(v)
    n = n or int(np.log2(len(v)))
    for k, a in enumerate(v):
        if abs(a) > 1e-12:
            print(f"  position {k} ({format(k, f'0{n}b')}): amplitude {np.round(a, 3)}, P = {abs(a)**2:.3f}")

from functools import reduce
kron = lambda *ms: reduce(np.kron, ms)
X = np.array([[0, 1], [1, 0]])
I2 = np.eye(2, dtype=int)

print("e0 ⊗ e0 =", kron(e0, e0).real.astype(int), "  (|00⟩)")
print("e1 ⊗ e0 ⊗ e1 =", kron(e1, e0, e1).real.astype(int), " (|101⟩ = e5)")

XII = kron(X, I2, I2)
XXX = kron(X, X, X)
labels = [format(k, "03b") for k in range(8)]
print("\nX ⊗ I ⊗ I:")
print(XII)
print("action:", {labels[k]: labels[int(np.argmax(XII[:, k]))] for k in range(8)})
print("\nX ⊗ X ⊗ X:", {labels[k]: labels[int(np.argmax(XXX[:, k]))] for k in range(8)})

e0 ⊗ e0 = [1 0 0 0]   (|00⟩)
e1 ⊗ e0 ⊗ e1 = [0 0 0 0 0 1 0 0]  (|101⟩ = e5)

X ⊗ I ⊗ I:
[[0 0 0 0 1 0 0 0]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 1]
 [1 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0 0]]
action: {'000': '100', '001': '101', '010': '110', '011': '111', '100': '000', '101': '001', '110': '010', '111': '011'}

X ⊗ X ⊗ X: {'000': '111', '001': '110', '010': '101', '011': '100', '100': '011', '101': '010', '110': '001', '111': '000'}

**Linearity on a superposition:** $(X \otimes I \otimes I)\,\frac{1}{\sqrt2}(e_0 + e_3) = \frac{1}{\sqrt2}(e_4 + e_7)$, i.e. the labels $|000\rangle + |011\rangle \mapsto |100\rangle + |111\rangle$.

**Watch the bit order:** PennyLane treats qubit 0 as the **first** bit (as here). Qiskit does the opposite: qubit 0 is the **last** bit. The same circuit in Qiskit shows "001" instead of "100".

**Simulator memory:** a state is $2^n$ complex numbers of 16 bytes each, hence the chart in the talk.

In [2]:
for n in (20, 25, 30, 35, 40, 50):
    nbytes = 16 * 2 ** n
    units = ["B", "KiB", "MiB", "GiB", "TiB", "PiB"]
    k = int(np.log2(nbytes) // 10)
    print(f"{n} qubits: 2^{n} amplitudes, {nbytes / 2 ** (10 * k):.0f} {units[k]}")

20 qubits: 2^20 amplitudes, 16 MiB
25 qubits: 2^25 amplitudes, 512 MiB
30 qubits: 2^30 amplitudes, 16 GiB
35 qubits: 2^35 amplitudes, 512 GiB
40 qubits: 2^40 amplitudes, 16 TiB
50 qubits: 2^50 amplitudes, 16 PiB

## 3. Superposition: a precise definition

> A **superposition** is a non-trivial linear combination of vectors of **a given basis**: the state has at least two non-zero coordinates in that basis.

Some consequences worth keeping straight:

- **"At least two" does not depend on the number of qubits.** One non-zero coordinate means the state is (up to phase) a single basis vector. For 3 qubits, $(e_0 + e_7)/\sqrt2$ has only 2 non-zero coordinates out of 8 and is a superposition.
- **Not all coordinates need to be non-zero.** The state $(1, 1, 1, 1)/2$ is a special case: an equal superposition of all basis states (the starting point of Grover's algorithm).
- **Physically there are $2^n$ states without superposition:** the basis vectors, up to global phase. For one qubit these are the poles of the Bloch sphere.
- **The notion depends on the basis.** Every vector is a linear combination of something (e.g. $e_0 = (u_+ + u_-)/\sqrt2$). $u_+$ is a superposition in the $z$ basis and a basis vector in the $x$ basis. When nobody says "in which basis", the computational basis is meant.
- **Watch normalisation:** $(1/2, 1/2)$ is not a state (norm $1/\sqrt2$); the correct counterpart is $(1/\sqrt2, 1/\sqrt2)$.

"A qubit in superposition is 0 and 1 at the same time" is a misleading sentence. Better: the qubit is in a state different from $e_0$ and from $e_1$, and a measurement in the $z$ basis gives a random outcome.

**The superposition principle** (the second meaning of the word) is a postulate: every normalised linear combination of states is a state, i.e. states form a vector space.

In [3]:
def is_superposition(v):
    return np.count_nonzero(np.abs(v) > 1e-12) >= 2

examples = {
    "(0,1,0,0)  |01⟩":          np.array([0, 1, 0, 0]),
    "(1,0,0,1)/√2  Bell":       np.array([1, 0, 0, 1]) / np.sqrt(2),
    "(1,0,1,0)/√2  |+⟩⊗|0⟩":    np.array([1, 0, 1, 0]) / np.sqrt(2),
    "(1,1,1,1)/2  |+⟩⊗|+⟩":     np.array([1, 1, 1, 1]) / 2,
    "(e0+e7)/√2, 3 qubits":     (np.eye(8)[0] + np.eye(8)[7]) / np.sqrt(2),
}
for name, v in examples.items():
    print(f"{name:>26}: non-zero = {np.count_nonzero(v)}, superposition? {is_superposition(v)}")

           (0,1,0,0)  |01⟩: non-zero = 1, superposition? False
        (1,0,0,1)/√2  Bell: non-zero = 2, superposition? True
     (1,0,1,0)/√2  |+⟩⊗|0⟩: non-zero = 2, superposition? True
      (1,1,1,1)/2  |+⟩⊗|+⟩: non-zero = 4, superposition? True
      (e0+e7)/√2, 3 qubits: non-zero = 2, superposition? True

## Summary

> The state of $n$ qubits is a unit vector in $\mathbb{C}^{2^n} = (\mathbb{C}^2)^{\otimes n}$. The basis vectors are $e_k$, where $k$ written in binary is the bit string (one-hot encoding). Gates on single qubits are Kronecker products with identities. A superposition is a non-trivial linear combination of vectors of a given basis: at least two non-zero coordinates, regardless of the number of qubits.

## Exercises

1. Write $e_1 \otimes e_1 \otimes e_0$ as a vector in $\mathbb{C}^8$. What is its label?
2. What does the matrix $I \otimes X$ (X on the second of two qubits) look like? Where does $e_0 = |00\rangle$ go?
3. How many non-zero coordinates does the state $H\!\otimes\!H\!\otimes\!H\, e_0$ have? Is it a superposition?